# AMD Voice SFT — crof.ai EQ-Matrix Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YUGOROU/amd-voice-sft/blob/feature/data-preprocessing/crof_pipeline.ipynb)

**Layer 1**: Domain rewrite with EQ-Matrix parameters via crof.ai (DeepSeek-V3)  
**Layer 2**: Quality scoring and threshold filtering via crof.ai (DeepSeek-V3)  

**Required Colab Secrets** (left panel > key icon):
- `CROF_API_KEY` — crof.ai API key
- `HF_TOKEN` — Hugging Face write token

In [ ]:
# Cell 1: Install & imports
!pip install openai datasets huggingface_hub tqdm -q

import collections
import json
import random
import time
from pathlib import Path

from datasets import Dataset
from google.colab import drive, userdata
from huggingface_hub import login
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# Cell 2: Config — API keys loaded from Colab Secrets

CROF_API_KEY = userdata.get("CROF_API_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")

client = OpenAI(
    api_key=CROF_API_KEY,
    base_url="https://crof.ai/v1",
)
MODEL = "deepseek-v4-flash"

EQ_MATRIX = {
    "condition": ["dementia", "alzheimer's"],
    "severity":  ["mild", "moderate", "severe"],
    "emotion":   ["calm", "anxious", "nostalgic", "agitated", "withdrawn"],
    "scenario":  [
        "repetitive_questions",
        "time_place_confusion",
        "family_memories",
        "daily_care",
        "social_interaction",
    ],
}

CHARACTER = {
    "name":        "Lumi",
    "personality": "warm, patient, gentle, slightly playful, never clinical",
    "speech_style": "short sentences, simple vocabulary, voice-optimized",
}

KEEP_THRESHOLD = 32        # out of 40
HF_REPO_ID     = "YUGOROU/amd-voice-sft-dataset"
DRIVE_DIR      = Path("/content/drive/MyDrive/amd-voice-sft")

print("Config loaded. Model:", MODEL)

In [ ]:
# Cell 3: Mount Google Drive and load preprocessed dataset
# Intermediate files are saved to Drive to survive session disconnects

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def load_unified_dataset(path="data/train.jsonl"):
    samples = []
    with open(path) as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"Loaded {len(samples):,} samples from {path}")
    return samples

samples = load_unified_dataset("data/train.jsonl")
print(f"Sample[0] roles: {[m['role'] for m in samples[0]['messages']]}")

In [ ]:
# Cell 4: Prompt templates and utility functions

REWRITE_PROMPT = """Rewrite the following conversation according to these parameters.

## Patient Profile
- Condition: {condition}
- Severity: {severity}
- Emotional State: {emotion}
- Scenario Type: {scenario}

## Scenario Behavior Guide
- repetitive_questions: The user asks a similar question 1-2 times during the conversation.
- time_place_confusion: The user is unsure what day, year, or where they are.
- family_memories: The user brings up a family member or past memory.
- daily_care: The conversation involves eating, sleeping, medication, or daily routines.
- social_interaction: Light casual conversation, the user may feel lonely or seek connection.

## Severity Behavior Guide
- mild: Slight forgetfulness, mostly coherent, occasional repetition.
- moderate: Noticeable confusion, repeats questions, needs gentle redirection.
- severe: Significant disorientation, very short responses, needs constant reassurance.

## Emotion Behavior Guide
- calm: Relaxed, cooperative tone.
- anxious: Worried, restless, needs reassurance.
- nostalgic: Reflective, mentions the past warmly.
- agitated: Frustrated or upset, needs de-escalation.
- withdrawn: Quiet, short answers, needs gentle encouragement.

## AI Companion Profile
- Name: {character_name}
- Personality: {character_personality}
- Speech rules: sentences under 20 words, simple vocabulary, voice-optimized, never clinical

## Rewriting Rules
1. Transform the user role into an elderly person with the above profile.
2. Transform the assistant role into {character_name}.
3. Inject scenario patterns naturally — do not force them unnaturally.
4. Remove all clinical, therapeutic, or formal language.
5. Keep each turn short and suitable for text-to-speech.
6. Preserve the emotional arc of the original conversation.
7. Every assistant turn MUST follow the 3-part output format below.
8. Output ONLY valid JSON in the exact format below.

## Assistant Output Format (STRICT — every assistant turn must follow this)
Each assistant content must have exactly 3 parts in order:

PART 1 — Avatar tag + first utterance (max 8 words, spoken immediately):
  [ACTION_TAG] [SHORT_FIRST_UTTERANCE]
  ACTION_TAG must be one of: [smile] [nod] [concerned] [gentle] [laugh]

PART 2 — Reasoning (never sent to TTS, never shown to user):
  <think>
  [2-3 sentences analyzing patient state and reasoning about best response]
  </think>

PART 3 — Final response (short, warm, voice-optimized):
  [FINAL_RESPONSE — under 25 words, no clinical language]

Example assistant content:
  "[smile] Oh, I remember that too...\n<think>\nUser is in a nostalgic state, mild severity. Engaging with the memory warmly will build trust. Keep it short and invite them to share more.\n</think>\nShe sounds like such a wonderful person. What's your favorite memory of her?"

## Original Conversation
{original_conversation}

## Output Format (JSON only, no markdown fences)
{{
  "messages": [
    {{"role": "system", "content": "You are {character_name}, a warm and patient AI companion for elderly people. Always respond with kindness, patience, and simplicity. Keep your responses short and clear. Never use clinical or formal language. Always start your response with an avatar action tag ([smile], [nod], [concerned], [gentle], or [laugh]) followed by a short first utterance, then your reasoning in <think> tags, then your final response."}},
    {{"role": "user", "content": "..."}},
    {{"role": "assistant", "content": "[ACTION_TAG] [first utterance]\\n<think>\\n[reasoning]\\n</think>\\n[final response]"}}
  ],
  "metadata": {{
    "condition": "{condition}",
    "severity": "{severity}",
    "emotion": "{emotion}",
    "scenario": "{scenario}"
  }}
}}"""

FILTER_PROMPT = """Evaluate the following AI companion dialogue for dementia care.

## Conversation
{conversation}

## Scoring Criteria (each 0-10)
1. empathy: Does the AI respond with genuine warmth and emotional attunement to the elderly user?
2. voice_suitability: Are sentences short (under 25 words)? Simple vocabulary? Natural when read aloud?
3. domain_fit: Does this feel like an authentic dementia companion interaction, not a therapy session?
4. format_compliance: Does every assistant turn correctly follow the [TAG] first utterance / <think> / final response structure?

## Scoring Rules
- Be strict. Reserve 9-10 for exceptional quality.
- Penalize heavily for: clinical language, long complex sentences, unnatural phrasing.
- Penalize for: ignoring the patient's emotional state, generic responses.
- Score format_compliance 0 if any assistant turn is missing the avatar tag, <think> block, or final response.

## Output Format (JSON only, no markdown fences)
{{
  "empathy": <int 0-10>,
  "voice_suitability": <int 0-10>,
  "domain_fit": <int 0-10>,
  "format_compliance": <int 0-10>,
  "total": <int sum>,
  "keep": <true if total >= 32 else false>,
  "reason": "<one sentence explaining the main strength or weakness>"
}}"""


def sample_eq_params():
    return {k: random.choice(v) for k, v in EQ_MATRIX.items()}


def format_conversation_for_prompt(messages):
    return "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages if m["role"] != "system"
    )


def call_crof(system_prompt, user_prompt, temperature=0.8, max_tokens=1500,
              reasoning_effort="medium", retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                reasoning_effort=reasoning_effort,
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{retries} in {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  Failed after {retries} attempts: {e}")
                return None


def parse_json_response(raw):
    if raw is None:
        return None
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e} | raw[:200]: {cleaned[:200]}")
        return None


print("Utilities ready.")

In [ ]:
# Cell 5: Layer 1 — EQ-Matrix rewrite pipeline

REWRITE_SYSTEM = (
    "You are an expert dialogue rewriter specializing in dementia care AI companions. "
    "Rewrite conversations into natural dialogues between an elderly person with dementia "
    "and a caring AI companion. "
    "Always output valid JSON only. No preamble, no explanation, no markdown fences."
)


def rewrite_sample(sample, eq_params):
    original_conv = format_conversation_for_prompt(sample["messages"])
    prompt = REWRITE_PROMPT.format(
        condition=eq_params["condition"],
        severity=eq_params["severity"],
        emotion=eq_params["emotion"],
        scenario=eq_params["scenario"],
        character_name=CHARACTER["name"],
        character_personality=CHARACTER["personality"],
        original_conversation=original_conv,
    )
    raw = call_crof(REWRITE_SYSTEM, prompt, temperature=0.8, max_tokens=1500, reasoning_effort="medium")
    return parse_json_response(raw)


rewritten_samples = []
failed_count = 0

for i, sample in enumerate(tqdm(samples, desc="Layer 1 rewrite")):
    eq_params = sample_eq_params()
    rewritten = rewrite_sample(sample, eq_params)
    if rewritten:
        rewritten_samples.append(rewritten)
    else:
        failed_count += 1

    if (i + 1) % 50 == 0:
        time.sleep(1)

print(f"Rewritten: {len(rewritten_samples):,} / {len(samples):,}  Failed: {failed_count}")

# Save intermediate file to Drive (session resilience)
intermediate_path = DRIVE_DIR / "rewritten_dataset.jsonl"
with open(intermediate_path, "w") as f:
    for s in rewritten_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print(f"Saved to {intermediate_path}")

In [ ]:
# Cell 6: Layer 2 — Quality filter

FILTER_SYSTEM = (
    "You are a quality evaluator for dementia care AI companion dialogues. "
    "Score conversations strictly and return ONLY valid JSON. No preamble, no explanation."
)


def filter_sample(sample):
    conv_str = format_conversation_for_prompt(sample["messages"])
    prompt = FILTER_PROMPT.format(conversation=conv_str)
    raw = call_crof(FILTER_SYSTEM, prompt, temperature=0.1, max_tokens=300, reasoning_effort="low")
    return parse_json_response(raw)


filtered_samples = []
discard_count = 0
score_records = []

for sample in tqdm(rewritten_samples, desc="Layer 2 filter"):
    scores = filter_sample(sample)
    if scores is None:
        discard_count += 1
        continue
    score_records.append(scores)
    if scores.get("keep", False):
        sample["quality_scores"] = scores
        filtered_samples.append(sample)
    else:
        discard_count += 1

kept_pct   = len(filtered_samples) / len(rewritten_samples) * 100 if rewritten_samples else 0
avg_score  = sum(s["total"] for s in score_records) / len(score_records) if score_records else 0
print(f"Kept: {len(filtered_samples):,} ({kept_pct:.1f}%)  Discarded: {discard_count}  Avg score: {avg_score:.1f}/40")

# Save to Drive
filtered_path = DRIVE_DIR / "filtered_dataset.jsonl"
with open(filtered_path, "w") as f:
    for s in filtered_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print(f"Saved to {filtered_path}")

In [ ]:
# Cell 7: Stats and sample inspection

conditions = [s["metadata"]["condition"] for s in filtered_samples]
severities = [s["metadata"]["severity"]  for s in filtered_samples]
emotions   = [s["metadata"]["emotion"]   for s in filtered_samples]
scenarios  = [s["metadata"]["scenario"]  for s in filtered_samples]

print("=== EQ-Matrix Distribution ===")
for label, data in [("Condition", conditions), ("Severity", severities),
                    ("Emotion", emotions), ("Scenario", scenarios)]:
    print(f"{label}: {dict(collections.Counter(data))}")

print("\n=== Random Sample ===")
sample = random.choice(filtered_samples)
for m in sample["messages"]:
    print(f"[{m['role'].upper()}]\n{m['content']}\n")
print("Metadata:", sample["metadata"])
print("Scores:",   sample.get("quality_scores", {}))

In [ ]:
# Cell 8: Push to Hugging Face Hub
# Note: Enable Gated access manually at
# https://huggingface.co/datasets/YUGOROU/amd-voice-sft-dataset/settings

login(token=HF_TOKEN)

hf_data = {
    "messages":       [s["messages"]               for s in filtered_samples],
    "metadata":       [s["metadata"]               for s in filtered_samples],
    "quality_scores": [s.get("quality_scores", {}) for s in filtered_samples],
}

dataset = Dataset.from_dict(hf_data)
dataset.push_to_hub(HF_REPO_ID, private=True)

print(f"Pushed {len(dataset):,} samples to {HF_REPO_ID}")